# Aggregation across seeds

Reads the per-seed tables in `results/seed<K>/tables/` and produces mean ± SD (ddof = 1) over seeds for every method, dataset and resolution.

- A run in which a representation produced **no valid clustering** is counted, not dropped: every cell reports `valid/runs`, and a row with 0 valid runs is printed as "no valid clustering".
- Rows from the DSBM approximation and the JODIE reimplementation are excluded.
- Deterministic representations (raw low-order, motif counts) have SD of about 0.
- Seeds change triad sampling, network initialisation (fusion, GRU, TGN), Louvain seeding and UMAP.

Outputs go to `results/aggregated/`. Quantities that the pipeline prints rather than saves (smoothness sweep, drift, transition table, bootstrap) are collected by `scripts/parse_run_logs.py`.

In [ ]:
import glob, re
import numpy as np, pandas as pd
from pathlib import Path
RES=Path('results'); OUT=RES/'aggregated'; OUT.mkdir(parents=True, exist_ok=True)
SEEDS=sorted(int(p.name[4:]) for p in RES.glob('seed*'))
rows=[];d4=[]
for k in SEEDS:
    for f in glob.glob(f'results/seed{k}/tables/table_headline_*.csv'):
        m=re.search(r'headline_(\w+?)_(\d+)h',f); t=pd.read_csv(f); t['seed']=k; t['Dataset']=m.group(1); t['Resolution_h']=int(m.group(2)); rows.append(t)
    for f in glob.glob(f'results/seed{k}/tables/table_D4_corrected_ablation_*.csv'):
        m=re.search(r'ablation_(\w+?)_(\d+)h',f); t=pd.read_csv(f); t['Dataset']=m.group(1); t['Resolution_h']=int(m.group(2)); t['seed']=k; d4.append(t)
H=pd.concat(rows); D=pd.concat(d4)
H=H[~H.Method.isin(['DSBM_APPROX_SPECTRAL','JODIE_SIMPLIFIED'])]
# long-format raw record of every run (for the repository)
L1=H[['seed','Dataset','Resolution_h','Method','Silhouette','DaviesBouldin','n_clusters']].assign(source='headline')
L2=D.rename(columns={'Model':'Method'})[['seed','Dataset','Resolution_h','Method','Silhouette','DaviesBouldin','n_clusters']].assign(source='D4_pipeline_controlled')
pd.concat([L1,L2]).to_csv(OUT/'all_runs_long.csv',index=False)

def stat(v):
    v=pd.Series(v,dtype=float); ok=v.notna(); n=len(v)
    if ok.sum()==0: return dict(valid=f'0/{n}',mean=np.nan,sd=np.nan)
    return dict(valid=f'{ok.sum()}/{n}',mean=v[ok].mean(),sd=v[ok].std(ddof=1) if ok.sum()>1 else np.nan)
SPEC=[('Motif-based','Baseline (raw features)',H,'Method','Motif-based'),
 ('Low-order (degree stats), raw','Baseline (raw features)',H,'Method','Low-order (degree stats)'),
 ('Community (Louvain)','Baseline (own protocol)',H,'Method','Community (Louvain)'),
 ('TGN (PyG reference)','Baseline (trained)',H,'Method','TGN_PYG_REFERENCE'),
 ('Low-order through pipeline','Pipeline-controlled',D,'Model','Low-order only (pipeline-controlled)'),
 ('Proposed: full multimodal (normalized loss)','Pipeline-controlled',D,'Model','S + Temp + Sem (normalized, mean-loss)'),
 ('Proposed: structural-only','Pipeline-controlled',D,'Model','S only')]
recs=[]
for ds in ('higgs','reddit'):
  for r in (1,6,12):
    for label,grp,df,col,key in SPEC:
        g=df[(df.Dataset==ds)&(df.Resolution_h==r)&(df[col]==key)].set_index('seed')
        sil=[g.Silhouette.get(k,np.nan) for k in SEEDS]; db=[g.DaviesBouldin.get(k,np.nan) for k in SEEDS]
        s=stat(sil); b=stat(db)
        recs.append(dict(Dataset=ds,Resolution_h=r,Group=grp,Method=label,valid=s['valid'],Sil_mean=s['mean'],Sil_sd=s['sd'],DB_mean=b['mean'],DB_sd=b['sd'],
            Silhouette=('no valid clustering' if np.isnan(s['mean']) else (f"{s['mean']:.3f} (1 run)" if np.isnan(s['sd']) else f"{s['mean']:.3f} ± {s['sd']:.3f}")),
            DB=('--' if np.isnan(b['mean']) else (f"{b['mean']:.3f} (1 run)" if np.isnan(b['sd']) else f"{b['mean']:.3f} ± {b['sd']:.3f}"))))
T=pd.DataFrame(recs); T.to_csv(OUT/'manuscript_tables_all.csv',index=False)
def tex(g,cap,lab):
    lines=[r'\begin{table}[ht]',r'\centering',r'\caption{'+cap+'}',r'\label{'+lab+'}',r'\begin{tabular}{llccc}',r'\hline',r'Method & Valid/runs & Silhouette $\uparrow$ & Davies--Bouldin $\downarrow$ \\'.replace('llccc','')]
    lines=[r'\begin{table}[ht]',r'\centering',r'\caption{'+cap+'}',r'\label{'+lab+'}',r'\begin{tabular}{lccc}',r'\hline',r'Method & Valid/runs & Silhouette $\uparrow$ & Davies--Bouldin $\downarrow$ \\',r'\hline']
    for _,x in g.iterrows():
        sil=x.Silhouette.replace('±',r'$\pm$'); db=x.DB.replace('±',r'$\pm$')
        lines.append(f"{x.Method.replace('&',r'\&')} & {x.valid} & {sil} & {db} \\\\")
    lines+= [r'\hline',r'\end{tabular}',r'\end{table}']
    return '\n'.join(lines)
for ds in ('higgs','reddit'):
  for r in (1,6,12):
    g=T[(T.Dataset==ds)&(T.Resolution_h==r)]
    cap=(f"{ds.capitalize()}, {r}-hour resolution. Mean $\\pm$ SD over five independent seeds (valid runs / total runs; invalid clusterings are not scored). "
         "Raw-feature baselines are deterministic (SD $\\approx$ 0); trained and pipeline-controlled rows vary with the seed.")
    (OUT/f'table_{ds}_{r}h.tex').write_text(tex(g,cap,f'tab:ms_{ds}_{r}h'))
# full D4 ablation (all subsets) incl. 0/5
D4s=[]
for (ds,r),g in D.groupby(['Dataset','Resolution_h']):
    for m,gg in g.groupby('Model'):
        s=stat(gg.set_index('seed').Silhouette.reindex(range(1,6))); b=stat(gg.set_index('seed').DaviesBouldin.reindex(range(1,6)))
        D4s.append(dict(Dataset=ds,Resolution_h=r,Model=m,valid=s['valid'],Silhouette=('no valid clustering' if np.isnan(s['mean']) else (f"{s['mean']:.3f} (1 run)" if np.isnan(s['sd']) else f"{s['mean']:.3f} ± {s['sd']:.3f}")),DB=('--' if np.isnan(b['mean']) else (f"{b['mean']:.3f} (1 run)" if np.isnan(b['sd']) else f"{b['mean']:.3f} ± {b['sd']:.3f}")),Sil_mean=s['mean'],Sil_sd=s['sd']))
pd.DataFrame(D4s).sort_values(['Dataset','Resolution_h','Sil_mean'],ascending=[True,True,False]).to_csv(OUT/'ablation_all_subsets.csv',index=False)
pd.set_option('display.width',200)
for ds,r in (('higgs',1),('reddit',6)):
    print(T[(T.Dataset==ds)&(T.Resolution_h==r)][['Method','valid','Silhouette','DB']].to_string(index=False))


In [ ]:
# printed-quantity tables (smoothness sweep, drift, transition table, bootstrap)
%run scripts/parse_run_logs.py results results/aggregated